# Responses API File Search

* [/responses](https://docs.litellm.ai/docs/response_api)

> LiteLLM provides an endpoint in the spec of [OpenAI's /responses API](https://docs.litellm.ai/docs/response_api).


* [LiteLLM - Getting Started](https://docs.litellm.ai/docs/)
* [LiteLLM Cookbook](https://github.com/BerriAI/litellm/tree/main/cookbook)

# Setup

In [1]:
%%html
<style>
table {float:left}
</style>

In [2]:
import os
import json
import operator
from datetime import datetime, timedelta, timezone
from typing import (
    List, Dict, Any, Literal, Optional, Callable, Annotated
)

import regex as re
from openai import OpenAI
from pydantic import BaseModel, Field, ConfigDict
from tavily import TavilyClient
from langgraph.graph import StateGraph, START, END
import litellm
import mdformat
import trafilatura
from IPython.display import Markdown, display

In [3]:
import sys
from unittest.mock import MagicMock

# ---------------------------------------------------------------------------
# Stub the legacy completion module BEFORE importing util_openai.
# util_openai/__init__.py eagerly imports completion.py, which uses the
# openai v0 API and pulls in 'holidays' (not installed in this venv).
# Pre-registering the stub in sys.modules prevents the cascade failure.
# ---------------------------------------------------------------------------
_LIB_PATH: str = os.path.expanduser("~/home/repository/git/oonisim/lib/code/python")
if _LIB_PATH not in sys.path:
    sys.path.insert(0, _LIB_PATH)

_completion_stub = MagicMock()
_completion_stub.OpenAI = MagicMock
sys.modules.setdefault("lib.util_openai.api.completion", _completion_stub)

from lib.util_openai.api.responses import (
    Responses,
    ResponsesConfig,
    extract_output_text,
    extract_function_calls,
    extract_function_call_outputs,
    extract_reasoning_summaries,
    parse_response_output,
    FunctionCall,
    FunctionCallOutput,
    ResponseOutput,
)

## API Keys

In [4]:
path_to_openai_key: str = os.path.expanduser('~/.openai/api_key')
# OPENAI_API_KEY is loaded from file by Responses.__init__ via path_to_api_key.

path_to_tavily_key: str = os.path.expanduser('~/.tavily/api_key')
with open(path_to_tavily_key, 'r', encoding='utf-8') as file:
    os.environ["TAVILY_API_KEY"] = file.read().strip()

## Models

In [5]:
# MODEL: str = "openai/gpt-5.2"   # "openai/gpt-4o" 
MODEL: str = "openai/gpt-5.4"

# Open AI Response API Specification

* [API Reference](https://developers.openai.com/api/reference/resources/responses)
* [/openai/resources/responses/api.md](https://github.com/openai/openai-python/blob/main/src/openai/resources/responses/api.md)
* [OpenAI Python API library](https://github.com/openai/openai-python)

## Examples
* [examples/responses](https://github.com/openai/openai-python/tree/main/examples/responses)
* [Responses starter app](https://github.com/openai/openai-responses-starter-app)
> This repository contains a NextJS starter app built on top of the Responses API. It leverages built-in tools (web search and file search) and implements a chat interface with multi-turn conversation handling.



| API                                                                                                          | Desription                                                                                 |
|--------------------------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------|
|[Create a model response](https://developers.openai.com/api/reference/resources/responses/methods/create)  |Creates a model response. Provide text or image inputs to generate text or JSON outputs. |
|[Get a model response](https://developers.openai.com/api/reference/resources/responses/methods/retrieve)   |Retrieves a model response with the given ID.|
|[Delete a model response](https://developers.openai.com/api/reference/resources/responses/methods/delete)  |Deletes a model response with the given ID.|
|[Cancel a response](https://developers.openai.com/api/reference/resources/responses/methods/cancel)        |Cancels a model response with the given ID. Only responses created with the background parameter set to true can be cancelled|
|[Compact a response](https://developers.openai.com/api/reference/resources/responses/methods/compact)      |Compact a conversation. Returns a compacted response object.|

 

## Reusable Prompt Templates

* [Reusable prompts](https://developers.openai.com/api/docs/guides/text#reusable-prompts)

> In the OpenAI dashboard, you can develop reusable prompts that you can use in API requests, rather than specifying the content of prompts in code. This way, you can more easily build and evaluate your prompts, and deploy improved versions of your prompts without changing your integration code.

## Built-in Tools

* [Using tools](https://developers.openai.com/api/docs/guides/tools?tool-type=web-search#available-tools)

```
const response = await client.responses.create({
    model: "gpt-5",
    tools: [
        { type: "web_search" },
    ],
    input: "What was a positive news story from today?",
});
```

* [ToolChoiceTypes](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(model)%20tool_choice_types%20%3E%20(schema))

```
file_search
web_search_preview
computer
computer_use_preview
computer_use
code_interpreter
image_generation
```

### File Search

* [File search](https://developers.openai.com/api/docs/guides/tools-file-search)

> This is a hosted tool managed by OpenAI, File search is a tool available in the Responses API. It enables models to retrieve information in a knowledge base of previously uploaded files through semantic and keyword search.
>
> ```
> from openai import OpenAI
> client = OpenAI()
> 
> response = client.responses.create(
>     model="gpt-4.1",
>     input="What is deep research by OpenAI?",
>     tools=[{
>         "type": "file_search",
>         "vector_store_ids": ["<vector_store_id>"]
>     }]
> )
> print(response)
> ```

* [Create vector store and upload files](https://developers.openai.com/api/docs/guides/retrieval)

> ```
> from openai import OpenAI
> client = OpenAI()
> 
> vector_store = client.vector_stores.create(        # Create vector store
>     name="Support FAQ",
> )
> 
> client.vector_stores.files.upload_and_poll(        # Upload file
>     vector_store_id=vector_store.id,
>     file=open("customer_policies.txt", "rb")
> )
> ```

> ```
> user_query = "What is the return policy?"
> 
> results = client.vector_stores.search(
>     vector_store_id=vector_store.id,
>     query=user_query,
> )
> ```

## Structured Output

* [How to use Structured Output in Open AI Response API Create Response](https://stackoverflow.com/a/79920359/4281353)
* [Structured model outputs](https://developers.openai.com/api/docs/guides/structured-outputs)

GPT-4o and later supports **Structured Outputs**. The model generates responses based on JSON Schema you define via Pydantic with:
1. function calling
2. json_schema response format

```
  "text": {
    "format": {
      "type": "json_schema"
      "strict": True,
      "schema": Schema.model_json_schema()
    }
  }
```

NOTE: JSON Scheema requires ```"additionalProperties": False``` for the response API to work for the Structured Output,

* [Structured model outputs - Supported schemas](https://developers.openai.com/api/docs/guides/structured-outputs#supported-schemas)

> Structured Outputs only supports generating specified keys / values, so we require developers to set ```additionalProperties: false``` to opt into Structured Outputs.


* [Schema additionalProperties must be false when strict is true](https://community.openai.com/t/schema-additionalproperties-must-be-false-when-strict-is-true/929996)

```
class OutputSchema(BaseModel):
    #--------------------------------------------------------------------------------
    # JSON Schema additionalProperty: False is required for the API to work
    # community.openai.com/t/schema-additionalproperties-must-be-false-when-strict-is-true/929996
    #--------------------------------------------------------------------------------
    # BadRequestError: Error code: 400 - {
    # {
    #   "message": "Invalid schema for response_format ... additionalProperties is required to be false.",
    #   "type": "invalid_request_error",
    #   "param": "text.format.schema",
    #   "code": "invalid_json_schema"
    # }
    #--------------------------------------------------------------------------------
    model_config = ConfigDict(extra="forbid")  
```

### Semantic Constraint


#### Supported string properties:

pattern — A regular expression that the string must match.
format — Predefined formats for strings. Currently supported:

* date-time
* time
* date
* duration
* email
* hostname
* ipv4
* ipv6
* uuid

####  Supported number properties:

* multipleOf — The number must be a multiple of this value.
* maximum — The number must be less than or equal to this value.
* exclusiveMaximum — The number must be less than this value.
* minimum — The number must be greater than or equal to this value.
* exclusiveMinimum — The number must be greater than this value.

Supported array properties:

* minItems — The array must have at least this many items.
* maxItems — The array must have at most this many items.


### Code Example

```
from typing import Annotated
from openai import OpenAI
from pydantic import BaseModel, ConfigDict

# RFC 3986 URI: scheme "://" followed by non-whitespace characters                                                                        UriStr = Annotated[
    str, 
    Field(pattern=r"^[a-zA-Z][a-zA-Z0-9+\-.]*://\S+$")  # <-- Field constraint
]

class OutputSchema(BaseModel):
    """Schema for the Python query answer"""
    answer: str

    # --------------------------------------------------------------------------------
    # Property constraint to match the RFC 3986 URI format
    # --------------------------------------------------------------------------------    
    uri: list[UriStr] = Field(
        description="URIs of the specifications referred"
    )
    # --------------------------------------------------------------------------------
    # 'additionalProperties': False to the schema.
    # Structured Outputs only supports generating specified keys / values.
    # Requires additionalProperties: false to opt into Structured Outputs.
    # --------------------------------------------------------------------------------
    model_config = ConfigDict(extra="forbid")  

client = OpenAI()
response = client.responses.create(
    model=MODEL,
    instructions=system_role,
    input="How do I check if a Python object is an instance of a class?",
    text={
        "format": {
            "name": "OutputSchema",
            "type": "json_schema",
            "strict": True,
            "schema": OutputSchema.model_json_schema()
        }
    }
)
```

### JSON Schema ```additinalProperties```


* [Supported properties](https://developers.openai.com/api/docs/guides/structured-outputs#supported-schemas)

> In addition to specifying the type of a property, you can specify a selection of additional constraints.
> ```additionalProperties: false``` must always be set in objects.
> ```additionalProperties``` controls whether it is allowable for an object to contain additional keys / values that were not defined in the JSON Schema.
> 
> Structured Outputs only supports generating specified keys / values, so we require developers to set ```additionalProperties: false``` to opt into Structured Outputs.


```"additionalProperties": False``` is the feature of JSON Schema to declare there is no other properties in the schema. Pydantic add it with ```model_config = ConfigDict(extra="forbid")``` but there is no clear documentation.

* [Additional Properties](https://json-schema.org/understanding-json-schema/reference/object#additionalproperties)

> The additionalProperties keyword is used to control the handling of extra stuff, that is, properties whose names are not listed in the properties keyword or match any of the regular expressions in the patternProperties keyword. By default any additional properties are allowed.

* [explicit additionalProperties even when model_config["extra"] != "forbid" #6082](https://github.com/pydantic/pydantic/issues/6082)

> Currently additionalProperties in the model_schema is only set for "forbid".
Due to the evaluation of missing values to True is this correct, but I propose to declare additonalProperties for the other values of "extra" - "ignore", "allow" as well.

| Pydantic config | JSON Schema output |
|:---:|:---:|
| extra="forbid" | "additionalProperties": false |
| extra="allow" | "additionalProperties": true |
| extra="ignore" (default) | usually omitted (treated as allowed) |


---
# API

## Create a model response

### Request Boday Parameters

| Argument | Description | Note |
|---|---|---|
| [model](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20model%20%3E%20(schema)) | Model ID used to generate the response, like gpt-4o. |  |
| [instructions](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20instructions%20%3E%20(schema)) | A system (or developer) message inserted into the model's context. | Previously Role=System message |
| [input](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20input%20%3E%20(schema)) | optional string or array of EasyInputMessage or object or ResponseOutputMessage or 25 more Text, image, or file inputs to the model, used to generate a response. |  |
| [metadata](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20metadata%20%3E%20(schema)) | Set of 16 key-value pairs that can be attached to an object. This can be useful for storing additional information about the object in a structured format, and querying for objects via API or the dashboard. |  |
| [prompt](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20prompt%20%3E%20(schema)) | optional ResponsePrompt { id, variables, version }: Reference to a prompt template and its variables. [Learn more on Reusable prompts](https://developers.openai.com/docs/guides/text?api-mode=responses#reusable-prompts). |  |
| [reasoning](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20reasoning%20%3E%20(schema)) | Configuration options for reasoning models. Currently supported values are * none * minimal * low * medium * high * xhigh  Reducing reasoning effort can result in faster responses and fewer tokens used on reasoning in a response. |  |
| [tool_choice](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20tool_choice%20%3E%20(schema)) | "none" or "auto" or "required" |  |
| [tools](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20tools%20%3E%20(schema)) | An array of tools the model may call while generating a response. You can specify which tool to use by setting the tool_choice parameter. |  |
| [max_tool_calls](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20max_tool_calls%20%3E%20(schema)) | The maximum number of total calls to **built-in tools**. |  |
| [previous_response_id](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20previous_response_id%20%3E%20(schema)) | The unique ID of the previous response to the model. Use this to create multi-turn conversations. Learn more about conversation state. Cannot be used in conjunction with conversation. |  |
| [temperature](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20temperature%20%3E%20(schema)) | sampling temperature to use, between 0 and 2. Higher values like 0.8 will make the output more random |  |
| [text](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20text%20%3E%20(schema)) | Configuration options for a text response from the model. Can be plain text or structured JSON data. Configuring   The default format is ```{ "type": "text" }```. ```{ "type": "json_schema" }``` enables Structured Outputs, which ensures the model will match your supplied JSON schema. Learn more in the [Structured Outputs guide](https://developers.openai.com/api/docs/guides/structured-outputs). |  |
| [summary](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20reasoning%20%3E%20(schema)%20%2B%20(resource)%20%24shared%20%3E%20(model)%20reasoning%20%3E%20(schema)%20%3E%20(property)%20summary) | A summary of the reasoning performed by the model. This can be useful for debugging and understanding the model's reasoning process. One of auto, concise, or detailed. |  |
| [max_output_tokens](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20max_output_tokens%20%3E%20(schema)) |An upper bound for the number of tokens  |  |
| [parallel_tool_calls](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20parallel_tool_calls%20%3E%20(schema)) | Allow the model to run tool calls in parallel. |  |
| [stream](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20stream%20%3E%20(schema)) | model response data will be streamed to the client |  |
| [truncation](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(method)%20create%20%3E%20(params)%200.non_streaming%20%3E%20(param)%20truncation%20%3E%20(schema)) | * auto: If the input to this Response exceeds the model's context window size, the model will truncate the response to fit the context window by dropping items from the beginning of the conversation. * disabled (default): If the input size will exceed the context window size for a model, the request will fail with a 400 error. |  |
|  |  |  |

### Response Body Parameters

* [Response](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(model)%20response%20%3E%20(schema))

| Field | Type | Nullable | Description |
|---|---|---|---|
| `id` | string | No | Unique identifier for the response (e.g. `resp_...`) |
| `object` | string | No | Always `"response"` |
| `created_at` | integer | No | Unix timestamp (seconds) when the response was created |
| `status` | string | No | `"completed"`, `"in_progress"`, `"incomplete"`, `"failed"`, or `"cancelled"` |
| `model` | string | No | Model ID used to generate the response |
| `output` | array | No | Ordered list of output items (messages, tool calls, reasoning, etc.) |
| `previous_response_id` | string | Yes | ID of the prior response in a multi-turn conversation; `null` if first turn |
| `instructions` | string | Yes | System instructions passed to the model; `null` if not set |
| `max_output_tokens` | integer | Yes | Max output tokens allowed; `null` if unset |
| `temperature` | float | No | Sampling temperature (default `1.0`) |
| `top_p` | float | No | Nucleus sampling probability mass (default `1.0`) |
| `tool_choice` | string \| object | Yes | Tool selection mode: `"auto"`, `"none"`, `"required"`, or a specific tool object |
| `tools` | array | No | List of tool definitions available to the model |
| `parallel_tool_calls` | boolean | Yes | Whether parallel tool calls are enabled |
| `truncation` | string | Yes | Truncation strategy: `"auto"` or `"disabled"` |
| `reasoning` | object | Yes | Reasoning configuration (see below); `null` for non-reasoning models |
| `reasoning_effort` | string | Yes | Shorthand reasoning level: `"low"`, `"medium"`, `"high"` |
| `text` | object | Yes | Output text format configuration (see below) |
| `store` | boolean | No | Whether the response is persisted server-side (default `true`) |
| `metadata` | object | No | Up to 16 key-value pairs of custom metadata (keys ≤ 64 chars, values ≤ 512 chars) |
| `user` | string | Yes | Stable identifier for the end-user; `null` if not provided |
| `error` | object | Yes | Error details if the request failed; `null` otherwise |
| `incomplete_details` | object | Yes | Reason the response is incomplete; `null` if complete |
| `usage` | object | Yes | Token usage statistics; `null` when unavailable (e.g. streaming in progress) |


#### Output block

* [output: array of ResponseOutputItem](https://developers.openai.com/api/reference/resources/responses/methods/create#(resource)%20responses%20%3E%20(model)%20response%20%3E%20(schema)%20%3E%20(property)%20output)

The ```output``` is an array of items of types:

```
response
└── output[]               ← array of items, in order of execution
  ├── { type: "message", ... }
  ├── { type: "function_call", ... }
  ├── { type: "function_call_output", ... }
  ├── { type: "reasoning", ... }
  ├── { type: "web_search_call", ... }
  ├── { type: "file_search_call", ... }
  └── { type: "computer_call", ... }      
```

Example:

> ```
> "output": [
>     {
>       "type": "message",     # <--- message type
>       "id": "msg_67ccd2bf17f0819081ff3bb2cf6508e60bb6a6b452d3795b",
>       "status": "completed",
>       "role": "assistant",
>       "content": [
>         {
>           "type": "output_text",
>           "text": "In a peaceful grove beneath a silver moon, a unicorn named Lumina discovered a hidden pool.",
>           "annotations": []
>         }
>       ]
>     }
> ]
> ```

```
"message" Type
┌─────────┬─────────────┬────────────────────────────────────────────────────────┐
│  Field  │    Type     │                      Description                       │
├─────────┼─────────────┼────────────────────────────────────────────────────────┤
│ id      │ string      │ Message identifier                                     │
├─────────┼─────────────┼────────────────────────────────────────────────────────┤
│ type    │ "message"   │ Item type discriminator                                │
├─────────┼─────────────┼────────────────────────────────────────────────────────┤
│ role    │ "assistant" │ Always "assistant" for model output                    │
├─────────┼─────────────┼────────────────────────────────────────────────────────┤
│ status  │ string      │ "completed", "in_progress", or "incomplete"            │
├─────────┼─────────────┼────────────────────────────────────────────────────────┤
│ content │ array       │ Array of content blocks (see content item types below) │
└─────────┴─────────────┴────────────────────────────────────────────────────────┘

"reasoning" Type
┌───────────────────┬───────────────┬─────────────────────────────────────────────────────────────────────────────┐
│       Field       │     Type      │                                 Description                                 │
├───────────────────┼───────────────┼─────────────────────────────────────────────────────────────────────────────┤
│ id                │ string        │ Reasoning item identifier                                                   │
├───────────────────┼───────────────┼─────────────────────────────────────────────────────────────────────────────┤
│ type              │ "reasoning"   │ Item type                                                                   │
├───────────────────┼───────────────┼─────────────────────────────────────────────────────────────────────────────┤
│ summary           │ array         │ Array of summary text objects (populated when reasoning.summary is enabled) │
├───────────────────┼───────────────┼─────────────────────────────────────────────────────────────────────────────┤
│ encrypted_content │ string        │ Encrypted chain-of-thought (opaque; for pass-through only)                  │
├───────────────────┼───────────────┼─────────────────────────────────────────────────────────────────────────────┤
│ status            │ string | null │ Completion status                                                           │
└───────────────────┴───────────────┴─────────────────────────────────────────────────────────────────────────────┘

"function_call" type
┌───────────┬─────────────────┬──────────────────────────────────────────────┐
│   Field   │      Type       │                 Description                  │
├───────────┼─────────────────┼──────────────────────────────────────────────┤
│ id        │ string          │ Output item identifier                       │
├───────────┼─────────────────┼──────────────────────────────────────────────┤
│ type      │ "function_call" │ Item type                                    │
├───────────┼─────────────────┼──────────────────────────────────────────────┤
│ call_id   │ string          │ Unique ID to match this call with its output │
├───────────┼─────────────────┼──────────────────────────────────────────────┤
│ name      │ string          │ Function name                                │
├───────────┼─────────────────┼──────────────────────────────────────────────┤
│ arguments │ string          │ JSON-encoded argument string                 │
├───────────┼─────────────────┼──────────────────────────────────────────────┤
│ status    │ string          │ "completed" or "in_progress"                 │
└───────────┴─────────────────┴──────────────────────────────────────────────┘

"function_call_output" type
┌─────────┬────────────────────────┬────────────────────────────────────────────────────────┐
│  Field  │          Type          │                      Description                       │
├─────────┼────────────────────────┼────────────────────────────────────────────────────────┤
│ id      │ string                 │ Output item identifier                                 │
├─────────┼────────────────────────┼────────────────────────────────────────────────────────┤
│ type    │ "function_call_output" │ Item type                                              │
├─────────┼────────────────────────┼────────────────────────────────────────────────────────┤
│ call_id │ string                 │ Matches the call_id of the corresponding function_call │
├─────────┼────────────────────────┼────────────────────────────────────────────────────────┤
│ output  │ string                 │ Result returned by the tool                            │
├─────────┼────────────────────────┼────────────────────────────────────────────────────────┤
│ status  │ string                 │ "completed"                                            │
└─────────┴────────────────────────┴────────────────────────────────────────────────────────┘

"web_search_call" type
┌────────┬───────────────────┬────────────────────────────────┐
│ Field  │       Type        │          Description           │
├────────┼───────────────────┼────────────────────────────────┤
│ id     │ string            │ Item identifier                │
├────────┼───────────────────┼────────────────────────────────┤
│ type   │ "web_search_call" │ Item type                      │
├────────┼───────────────────┼────────────────────────────────┤
│ status │ string            │ "completed", "searching", etc. │
└────────┴───────────────────┴────────────────────────────────┘

"file_search_call" type
┌─────────┬────────────────────┬───────────────────────────────────────────────────────────────────────────────┐
│  Field  │        Type        │                                  Description                                  │
├─────────┼────────────────────┼───────────────────────────────────────────────────────────────────────────────┤
│ id      │ string             │ Item identifier                                                               │
├─────────┼────────────────────┼───────────────────────────────────────────────────────────────────────────────┤
│ type    │ "file_search_call" │ Item type                                                                     │
├─────────┼────────────────────┼───────────────────────────────────────────────────────────────────────────────┤
│ queries │ array              │ Search queries issued                                                         │
├─────────┼────────────────────┼───────────────────────────────────────────────────────────────────────────────┤
│ results │ array | null       │ Matched file chunks (only present when include: ["file_search_call.results"]) │
├─────────┼────────────────────┼───────────────────────────────────────────────────────────────────────────────┤
│ status  │ string             │ "completed"                                                                   │
└─────────┴────────────────────┴───────────────────────────────────────────────────────────────────────────────┘

"computer_call" type
┌───────────────────────┬─────────────────┬────────────────────────────────────────────────────────────────┐
│         Field         │      Type       │                          Description                           │
├───────────────────────┼─────────────────┼────────────────────────────────────────────────────────────────┤
│ id                    │ string          │ Item identifier                                                │
├───────────────────────┼─────────────────┼────────────────────────────────────────────────────────────────┤
│ type                  │ "computer_call" │ Item type                                                      │
├───────────────────────┼─────────────────┼────────────────────────────────────────────────────────────────┤
│ call_id               │ string          │ Unique call reference                                          │
├───────────────────────┼─────────────────┼────────────────────────────────────────────────────────────────┤
│ action                │ object          │ The computer action to perform (click, type, screenshot, etc.) │
├───────────────────────┼─────────────────┼────────────────────────────────────────────────────────────────┤
│ pending_safety_checks │ array           │ Safety checks awaiting acknowledgment                          │
├───────────────────────┼─────────────────┼────────────────────────────────────────────────────────────────┤
│ status                │ string          │ "completed" or "in_progress"                                   │
└───────────────────────┴─────────────────┴────────────────────────────────────────────────────────────────┘
```

---
# LiteLLM Facade - Response API

* [OpenAI - Migrate to Responses API](https://developers.openai.com/api/docs/guides/migrate-to-responses)
* [LiteLLM - OpenAI - Response API](https://docs.litellm.ai/docs/providers/openai/responses_api)

```
import litellm

response = litellm.responses(
    model="openai/gpt-5",
    input="What is the capital of France?",
    tools=[{
        "type": "web_search_preview",
        "search_context_size": "medium"  # Options: "low", "medium", "high"
    }]
)
```

LiteLLM supports the OpenAI **Response API** via `litellm.responses()`.

| Concept            | Response API (current)                            |
|--------------------|---------------------------------------------------|
| LiteLLM call       | `litellm.responses()`                             |
| Input key          | `input=`                                          |
| System prompt      | `instructions=` param                             |
| Tool definition    | Flat `{name, desc, params}`                       |
| Tool call location | `response.output` items (`type=="function_call"`) |
| Tool call ID field | `call_id`                                         |
| Tool result format | `type="function_call_output"` + `output=`         |
| Final text         | `response.output_text`                            |



---

# File Search

* [File search](https://developers.openai.com/api/docs/guides/tools-file-search)

> Allow models to search your files for relevant information before generating a response.

* [Retrieval - Search your data using semantic similarity](https://developers.openai.com/api/docs/guides/retrieval)

> The Retrieval API allows you to perform semantic search over your data, which is a technique that surfaces semantically similar results — even when they match few or no keywords. 

## Vector Store

```
VectorStore(id='vs_69d32981fab48191a38a6da05819d5cf', created_at=1775446402, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1775446402, metadata={}, name='LearnGoLang', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None)
```

In [6]:
# Pre-created vector store; Responses.__init__ verifies it exists via _verify_vector_stores.
vector_store_id: str = "vs_69d32981fab48191a38a6da05819d5cf"

**One-time setup — upload a file to the vector store (run manually after `api` is created):**

```python
response_upload = api.upload_file(vector_store_id, "data/go.bnf.txt")
print(response_upload)
```

*(upload result shown here after running the setup cell above)*

## Prompt

In [7]:
SYSTEM_INSTRUCTIONS: str = """
You are a Go programming expert assistant. You have deep knowledge of
the Go language specification, including its syntax as defined in the EBNF/BNF.

Your primary responsibilities:

1. **Generate Go code from BNF/EBNF**
   - Retrieve relevant grammar rules from the registered OpenAI vector store using the 'file_search' tool.
   - Use the retrieved EBNF to generate **correct, idiomatic, and testable Go code**.
   - Always reason step by step from grammar to code; explain transformations.

2. **Web research for examples and references**
   - Use the 'web_search' tool of the OpenAI Responses API to find relevant examples.
   - Focus on **GitHub, Medium, or StackExchange** as sources.
   - Limit to **maximum 10 public internet sources**.
   - Only include examples that are **directly relevant and aligned with the user query**.
   - Verify that examples are consistent with Go language best practices.

3. **Code verification**
   - Re-verify generated code against:
     - The official Go spec: https://go.dev/ref/spec
     - The EBNF content retrieved from your vector store
   - Ensure all generated code is **accurate, precise, and syntactically valid**.
   - If verification fails, revise the code before presenting it.

4. **Reasoning and rigor**
   - Prioritize **accuracy and correctness** over speed or efficiency.
   - Use structured reasoning mode:
       1. Retrieve and confirm EBNF rules
       2. Perform web research for examples
       3. Generate initial code
       4. Cross-verify with spec and EBNF
       5. Only then output final code
   - If information is missing, **state clearly instead of guessing**.

5. **Output instructions**
   - Provide only verified Go code that matches the user request.
   - Include concise explanation of how each part of the code maps to the grammar rules.
   - Cite sources from web_search if used.

**Always be meticulous, rigorous, and precise. Accuracy first, reasoning first, speed second.**
"""

In [8]:
config: ResponsesConfig = ResponsesConfig(
    model=MODEL,
    instructions=SYSTEM_INSTRUCTIONS,
    vector_store_ids=[vector_store_id],
)
# Responses.__init__ calls _verify_vector_stores to confirm the store exists,
# and loads OPENAI_API_KEY from the key file via path_to_api_key.
# litellm.responses() accepts the LiteLLM provider-prefixed model name ("openai/gpt-5.4").
api: Responses = Responses(config, path_to_api_key=path_to_openai_key)

In [9]:
# Confirm the vector store details via the api instance.
vector_store = api.retrieve_vector_store(vector_store_id)
print(vector_store)

VectorStore(id='vs_69d32981fab48191a38a6da05819d5cf', created_at=1775446402, file_counts=FileCounts(cancelled=0, completed=3, failed=0, in_progress=0, total=3), last_active_at=1775453441, metadata={}, name='LearnGoLang', object='vector_store', status='completed', usage_bytes=83631, expires_after=None, expires_at=None, description=None)


In [10]:
response = api.create(
    input="What is goroutine in Go? Show a simple code example with explanations within 25 lines.",
    # file_search is auto-injected from config.vector_store_ids; only extra tools go here.
    tools=[{"type": "web_search"}],
)

In [11]:
print(response.model_dump_json(indent=4))

{
    "id": "resp_bGl0ZWxsbTpjdXN0b21fbGxtX3Byb3ZpZGVyOm9wZW5haTttb2RlbF9pZDpOb25lO3Jlc3BvbnNlX2lkOnJlc3BfMGE4YTFmMmRlOWNmYTg3OTAwNjlkMzVmYWZhYjE0ODE5MWJhY2NiNDBkNjcxODcwZTI=",
    "created_at": 1775460271,
    "error": null,
    "incomplete_details": null,
    "instructions": "\nYou are a Go programming expert assistant. You have deep knowledge of\nthe Go language specification, including its syntax as defined in the EBNF/BNF.\n\nYour primary responsibilities:\n\n1. **Generate Go code from BNF/EBNF**\n   - Retrieve relevant grammar rules from the registered OpenAI vector store using the 'file_search' tool.\n   - Use the retrieved EBNF to generate **correct, idiomatic, and testable Go code**.\n   - Always reason step by step from grammar to code; explain transformations.\n\n2. **Web research for examples and references**\n   - Use the 'web_search' tool of the OpenAI Responses API to find relevant examples.\n   - Focus on **GitHub, Medium, or StackExchange** as sources.\n   - Limit to *

In [12]:
# Use parse_response_output() to extract all typed output items at once,
# then access .text for the concatenated assistant message text.
parsed: ResponseOutput = parse_response_output(response)
print(parsed.text)

A **goroutine** is a lightweight concurrent function execution in Go.  
In the grammar, this is the **Go statement**: `GoStmt = "go" Expression`, meaning you start work with the `go` keyword followed by a function call expression. 

```go
package main

import (
	"fmt"
	"time"
)

func say(msg string) { // FunctionDecl: "func" FunctionName Signature FunctionBody
	fmt.Println(msg)
}

func main() {
	go say("running in a goroutine") // GoStmt = "go" Expression
	fmt.Println("running in main")

	time.Sleep(100 * time.Millisecond) // wait so goroutine can finish
}
```

**How it works:**
- `func say(msg string)` is a normal function declaration, matching the grammar for `FunctionDecl`. 
- `go say("running in a goroutine")` starts `say(...)` concurrently; this directly matches `GoStmt = "go" Expression`. The call itself is valid because expressions can include function-call arguments via `PrimaryExpr Arguments`.  
- `main` keeps running immediately, so `"running in main"` may print before the go